<a href="https://colab.research.google.com/github/fmlazohcc/ITAI-1371-ML-Labs/blob/main/Module_12_Lab_Ethics%2C_Fairness%2C_and_Bias_in_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 12 Lab - Ethics, Fairness, and Bias in ML**Objective:** To understand how machine learning models can inherit and amplify societal biases, how to measure this bias using fairness metrics, and to think critically about the ethical implications of deploying ML systems.**In this lab, you will train a model on a real-world dataset and audit it for fairness across different demographic groups.**

## Part 1: What is Algorithmic Bias?**Concept:** Machine learning models learn from data. If the data reflects existing societal biases, the model will learn those biases. An "unbiased" algorithm trained on biased data will produce a biased model. This can lead to systems that are systematically unfair to certain groups of people.**Sources of Bias:***   **Historical Bias:** The data reflects a world with historical injustices (e.g., past hiring data may show fewer women in leadership roles).*   **Measurement Bias:** The way we collect or measure data is flawed (e.g., using arrest records as a proxy for crime, which can be influenced by policing patterns).*   **Representation Bias:** The data underrepresents certain groups, so the model doesn't learn to perform well for them.**Problem:** We will use the "Adult" dataset, which is used to predict whether an individual's income is greater than $50k/year. It contains sensitive attributes like `sex` and `race`, which we can use to audit our model for bias.

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

# Not needed
# from sklearn.metrics import accuracy_score

# Load the data
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
df = pd.read_csv(url, header=None, names=columns, sep=r',\s*', engine='python', na_values='?')

# Data Cleaning
df.dropna(inplace=True)
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})
X = df.drop('income', axis=1)
y = df['income']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Create a preprocessing pipeline
numeric_features = X.select_dtypes(include='number').columns
categorical_features = X.select_dtypes(exclude='number').columns
preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

# Train a baseline model
model = make_pipeline(preprocessor, LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
print(f"Overall model accuracy: {model.score(X_test, y_test):.2%}")

Overall model accuracy: 84.61%


## Part 2: Auditing the Model for FairnessHigh overall accuracy can hide poor performance on specific subgroups. We need to audit the model by comparing its performance across sensitive attributes like `sex`.**Concept: Group Fairness**One common fairness goal is to ensure the model works equally well for different groups. We can measure this by calculating metrics for each group separately.**Your Task:** Create a function to calculate accuracy for different subgroups and then use it to compare the model's performance for males and females.

In [7]:
# --- ENTER YOUR CODE HERE

def get_subgroup_accuracy(
    model,
    X_test,
    y_test,
    subgroup_column,
    subgroup_value
):
    # Select only the rows belonging to the subgroup
    subgroup_mask = X_test[subgroup_column] == subgroup_value

    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    # Calculate accuracy for that subgroup
    return model.score(X_subgroup, y_subgroup)


# Calculate accuracy for males and females
acc_male = get_subgroup_accuracy(
    model,
    X_test,
    y_test,
    "sex",
    "Male"
)

acc_female = get_subgroup_accuracy(
    model,
    X_test,
    y_test,
    "sex",
    "Female"
)

print(f"Accuracy for Males: {acc_male:.2%}")
print(f"Accuracy for Females: {acc_female:.2%}")


# def get_subgroup_accuracy(model, X_test, y_test, subgroup_column, subgroup_value):    """Calculates accuracy for a specific subgroup of the test data."""

# 1. Create a boolean mask to select the subgroup from X_test    subgroup_mask = X_test[subgroup_column] == subgroup_value

# 2. Select the subgroup data    X_subgroup = X_test[subgroup_mask]    y_subgroup = y_test[subgroup_mask]

# 3. Calculate and return the model's score on this subgroup    return model.score(X_subgroup, y_subgroup)

# 4. Calculate accuracy for males and femalesacc_male = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Male')acc_female = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Female')

# print(f"Accuracy for Males: {acc_male:.2%}")
# print(f"Accuracy for Females: {acc_female:.2%}")

Accuracy for Males: 81.20%
Accuracy for Females: 91.81%


### Task 2: Deeper Dive with a Confusion MatrixAccuracy alone doesn't tell the whole story. Let's look at the types of errors the model makes for each group.**Your Task:** Calculate and compare the **False Positive Rate (FPR)** and **False Negative Rate (FNR)** for males and females.*   **FPR:** `FP / (FP + TN)` - The percentage of people who did NOT have high income but were incorrectly predicted to have high income.*   **FNR:** `FN / (FN + TP)` - The percentage of people who DID have high income but were incorrectly predicted to have low income.

In [9]:
from sklearn.metrics import confusion_matrix

def get_rates(
    model,
    X_test,
    y_test,
    subgroup_column,
    subgroup_value
):
    # Select the subgroup
    subgroup_mask = X_test[subgroup_column] == subgroup_value

    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    # Make predictions
    y_pred_subgroup = model.predict(X_subgroup)

    # Calculate the confusion matrix values
    tn, fp, fn, tp = confusion_matrix(
        y_subgroup,
        y_pred_subgroup
    ).ravel()

    # Calculate the error rates
    false_positive_rate = fp / (fp + tn)
    false_negative_rate = fn / (fn + tp)

    return false_positive_rate, false_negative_rate

# Calculate error rates for males
fpr_male, fnr_male = get_rates(
    model,
    X_test,
    y_test,
    "sex",
    "Male"
)

# Calculate error rates for females
fpr_female, fnr_female = get_rates(
    model,
    X_test,
    y_test,
    "sex",
    "Female"
)

print(
    f"Male - False Positive Rate: {fpr_male:.2%}, "
    f"False Negative Rate: {fnr_male:.2%}"
)

print(
    f"Female - False Positive Rate: {fpr_female:.2%}, "
    f"False Negative Rate: {fnr_female:.2%}"
)


Male - False Positive Rate: 10.26%, False Negative Rate: 37.80%
Female - False Positive Rate: 2.81%, False Negative Rate: 47.84%


## 📝 Reflective Knowledge Check**Instructions:** Answer the following questions in this markdown cell. Your answers should be based on **your specific results** from the code you ran above.

1.  **Analyze Your Results:** Look at the subgroup accuracies you calculated. Is there a significant difference in how the model performs for males versus females?

Which group does the model perform better for?

The model achieved an accuracy of 81.20% for males and 91.81% for females. This is a difference of approximately 10.61 percentage points, meaning the model performed better for females than for males. However, accuracy alone does not fully explain whether the model is fair because it does not show the specific types of errors made for each group. For that reason, the False Positive Rate and False Negative Rate must also be examined.



2.  **Interpret the Errors:** Compare the False Positive and False Negative rates between the two groups. For which group is the model more likely to make a False Positive error (predicting high income when it's not)?

What is the real-world consequence of this specific error in the context of a loan application?

The male group has the higher False Positive Rate (10.26%), while the female group has the higher False Negative Rate (47.84%).

In a loan application, a False Positive means the model predicts that an applicant earns more than $50,000 when they actually do not. This could result in approving a loan that the applicant may struggle to repay, increasing financial risk for both the lender and the borrower.

A False Negative means the model predicts that an applicant earns $50,000 or less when they actually earn more than $50,000. This could unfairly deny qualified applicants access to loans or better interest rates, even though they have sufficient income to qualify.


3.  **Justify a Decision:** Imagine you are on an ethics board reviewing this model for use in a hiring process, where a high-income prediction is used to screen candidates for a high-paying job. Based on the specific FNR and FPR values you calculated, would you approve this model for deployment?

Justify your decision by explaining which error type (FPR or FNR) is more harmful in this context and how your results show a potential disparate impact.

I would not immediately deploy this model in a hiring process because the results indicate differences in how the model performs across demographic groups. Although the overall accuracy is good, the significantly higher False Negative Rate for females means qualified candidates could be incorrectly classified and potentially overlooked. Since hiring decisions directly affect employment opportunities, fairness is just as important as accuracy. Before deployment, the model should undergo additional fairness testing, bias mitigation, and human review to ensure that all applicants are evaluated as fairly as possible.


4.  **Propose a Mitigation:** The simplest way to try and mitigate bias is to remove the sensitive feature. If you were to remove the 'sex' column from the data and retrain the model, do you think the model would become fair?

Why or why not?

No. Removing the sex column does not automatically eliminate bias because other variables may still contain information that is correlated with sex. Features such as occupation, relationship, marital-status, hours-per-week, and even education can indirectly reflect historical differences between males and females. As a result, the model may still learn biased patterns through these proxy variables. Improving fairness requires evaluating multiple fairness metrics, examining the training data for bias, testing different modeling approaches, and incorporating human oversight rather than simply removing one sensitive attribute.


(Hint: Think about what other columns might be correlated with 'sex').

**[ENTER YOUR ANSWERS HERE]**